# 📊 Topic 03: PySpark DataFrames, Schemas & Transformations

## 1. RDDs vs DataFrames
- **RDDs:** Unstructured, functional API, low-level Python object overhead, no query optimizer.
- **DataFrames:** Named columns, structured schema, powered by **Catalyst Optimizer** and **Tungsten Memory Management**.

---

## 2. Narrow vs Wide Dependencies
- **Narrow Dependency:** Each partition of the parent RDD is used by at most one partition of the child RDD (`map`, `filter`). **No Shuffling required!**
- **Wide Dependency:** Multiple child partitions depend on data from a single parent partition (`groupByKey`, `reduceByKey`, `join`). **Requires Network Shuffling!**

---

## 3. Hands-on: Defining Explicit Schemas & DataFrame Transformations


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import pyspark.sql.functions as F

spark = SparkSession.builder.master("local[*]").appName("DataFrames_Demo").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 1. Define Explicit Schema using StructType (Best practice for production)
schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("user_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("items_count", IntegerType(), True)
])

# Dummy dataset
raw_data = [
    ("T1001", "U101", "Electronics", 299.99, 1),
    ("T1002", "U102", "Clothing", 45.50, 3),
    ("T1003", "U101", "Electronics", 120.00, 2),
    ("T1004", "U103", "Groceries", 85.20, 5),
    ("T1005", "U102", "Electronics", 899.00, 1),
]

# Create DataFrame with schema
df = spark.createDataFrame(raw_data, schema=schema)

print("📋 DataFrame Schema:")
df.printSchema()

print("\n🔍 Transformation 1: Filter & Add Calculated Column (Narrow Dependency)")
df_transformed = df.filter(F.col("amount") > 50.0) \
                   .withColumn("avg_item_cost", F.round(F.col("amount") / F.col("items_count"), 2))

df_transformed.show()

print("\n📊 Transformation 2: GroupBy Category Aggregations (Wide Dependency - Shuffle)")
df_grouped = df.groupBy("category").agg(
    F.count("transaction_id").alias("total_txns"),
    F.sum("amount").alias("total_revenue"),
    F.avg("amount").alias("avg_order_value")
)

df_grouped.show()
